<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/09-math-in-code/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9: Solutions

Try the exercises first. The notes after each solution explain the maths behind it and the bug each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_stable_softmax(f):
    z2 = np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]])

    def unchanged():
        z = np.array([3.0, 1.0, 2.0])
        f(z)
        return z.tolist()

    _scenarios("stable_softmax", [
        ("small scores", lambda: f(np.array([0.0, 1.0, 2.0])), np.array([0.09003057, 0.24472847, 0.66524096]), None),
        ("large scores don't overflow", lambda: f(np.array([1000.0, 1001.0, 1002.0])),
         np.array([0.09003057, 0.24472847, 0.66524096]), "subtract the maximum before np.exp"),
        ("very negative scores don't give nan", lambda: f(np.array([-1000.0, -1000.0])), np.array([0.5, 0.5]),
         "exp(-1000) is 0; subtracting the maximum fixes this too"),
        ("2D: each row on its own", lambda: f(z2),
         np.array([[0.09003057, 0.24472847, 0.66524096], [1 / 3, 1 / 3, 1 / 3]]),
         "each row needs its own maximum: z.max(axis=-1, keepdims=True)"),
        ("2D rows sum to 1", lambda: f(np.arange(12.0).reshape(3, 4)).sum(axis=1), np.ones(3), "sum along axis=-1 with keepdims=True"),
        ("input is not modified", unchanged, [3.0, 1.0, 2.0], "z - z.max() makes a new array; z -= ... changes the caller's"),
    ])


def check_log_sum_exp(f):
    rng = np.random.default_rng(0)
    small = rng.normal(size=6)
    _scenarios("log_sum_exp", [
        ("matches the direct formula for small values", lambda: f(small), float(np.log(np.sum(np.exp(small)))), None),
        ("two zeros give log 2", lambda: f(np.array([0.0, 0.0])), math.log(2), None),
        ("large values don't overflow", lambda: f(np.array([1000.0, 1000.0])), 1000 + math.log(2),
         "use c + log(sum(exp(z - c))) with c = max(z)"),
        ("very negative values don't give -inf", lambda: f(np.array([-1000.0, -1000.0])), -1000 + math.log(2),
         "exp(-1000) underflows to 0; the max trick fixes it"),
        ("a single value", lambda: f(np.array([5.0])), 5.0, None),
        ("returns a float", lambda: type(f(np.array([1.0, 2.0]))) is float, True, "wrap the result in float(...)"),
    ])


def check_central_difference(f):
    _scenarios("central_difference", [
        ("derivative of sin at 1 is cos(1)", lambda: abs(f(math.sin, 1.0) - math.cos(1.0)) < 1e-9, True,
         "(f(x + h) - f(x - h)) / (2 * h)"),
        ("x cubed at 2 with h = 0.1 (error is exactly h squared)", lambda: f(lambda x: x ** 3, 2.0, 0.1), 12.01,
         "12.61 would mean a forward difference; check the brackets around (2 * h)"),
        ("uses the default h", lambda: abs(f(lambda x: x ** 3, 2.0) - 12.0) < 1e-8, True, None),
        ("a straight line has a constant slope", lambda: f(lambda x: 3 * x - 7, -4.0), 3.0, None),
    ])


def check_cosine_similarity(f):
    _scenarios("cosine_similarity", [
        ("same direction", lambda: f(np.array([1.0, 2.0, 3.0]), np.array([2.0, 4.0, 6.0])), 1.0, None),
        ("opposite directions", lambda: f(np.array([1.0, -1.0]), np.array([-3.0, 3.0])), -1.0, None),
        ("perpendicular", lambda: f(np.array([1.0, 0.0]), np.array([0.0, 5.0])), 0.0, None),
        ("a known angle (45 degrees)", lambda: f(np.array([1.0, 0.0]), np.array([1.0, 1.0])), 1 / math.sqrt(2), None),
        ("works with lists of ints", lambda: f([3, 4], [4, 3]), 0.96, None),
        ("zero vector raises", lambda: f(np.zeros(3), np.ones(3)), Raises(ValueError),
         "check both norms before dividing"),
        ("different lengths raise", lambda: f(np.ones(3), np.ones(4)), Raises(ValueError), None),
        ("returns a float", lambda: type(f(np.ones(2), np.ones(2))) is float, True, "wrap the result in float(...)"),
    ])


def check_numerical_gradient(f):
    def bowl(p):
        return p[0] ** 2 + 3 * p[1] ** 2

    W = np.array([[0.5, -1.0], [2.0, 0.25]])

    def unchanged():
        x = np.array([0.1, 0.7, -0.3])
        before = x.copy()
        f(lambda v: np.sum(np.sin(v)), x)
        return bool(np.array_equal(x, before))

    _scenarios("numerical_gradient", [
        ("a vector", lambda: f(bowl, np.array([1.0, -2.0])), np.array([2.0, -12.0]), None),
        ("a matrix of weights, same shape back", lambda: f(lambda m: np.sum(m ** 2), W), 2 * W,
         "loop over np.ndindex(x.shape) so any shape works"),
        ("integer input", lambda: f(lambda v: np.sum(v ** 2) / 4, np.array([1, 2])), np.array([0.5, 1.0]),
         "work on a float copy: np.array(x, dtype=float)"),
        ("the input is left exactly as it was", unchanged, True,
         "x[i] += h then x[i] -= h doesn't always restore the exact value; work on a copy and save the original"),
        ("matches a known gradient", lambda: f(lambda v: np.sum(np.sin(v)), np.array([0.0, 1.0])),
         np.array([1.0, math.cos(1.0)]), None),
    ])


def check_gradient_descent(f):
    def grad_sq(x):
        return 2 * x

    def unchanged():
        x0 = np.array([3.0, -1.0])
        f(grad_sq, x0, 0.1, 5)
        return x0.tolist()

    _scenarios("gradient_descent", [
        ("halving each step: x squared, lr 0.25", lambda: f(grad_sq, [3.0], 0.25, 3),
         np.array([[3.0], [1.5], [0.75], [0.375]]), None),
        ("path keeps every step, not copies of the last one", lambda: f(grad_sq, np.array([4.0, -2.0]), 0.25, 2),
         np.array([[4.0, -2.0], [2.0, -1.0], [1.0, -0.5]]),
         "x -= ... changes the same array each time; use x = x - ... and append a copy"),
        ("zero steps returns just the start", lambda: f(grad_sq, [1.0, 2.0], 0.1, 0), np.array([[1.0, 2.0]]), None),
        ("shape is (steps + 1, d)", lambda: f(grad_sq, [1.0, 2.0, 3.0], 0.1, 10).shape, (11, 3), None),
        ("a stretched bowl converges with lr 0.09", lambda: bool(np.allclose(
            f(lambda p: np.array([2 * p[0], 20 * p[1]]), [-4.0, 1.5], 0.09, 300)[-1], 0, atol=1e-6)), True, None),
        ("x0 is not modified", unchanged, [3.0, -1.0], "start from a copy: np.array(x0, dtype=float)"),
    ])


def check_cross_entropy(f):
    probs = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.3, 0.3, 0.4]])
    labels = np.array([0, 1, 2])
    expected = float(-np.mean(np.log([0.7, 0.8, 0.4])))
    _scenarios("cross_entropy", [
        ("perfect predictions give 0", lambda: f(np.eye(3), [0, 1, 2]), 0.0, None),
        ("uniform predictions give log(k)", lambda: f(np.full((4, 3), 1 / 3), [0, 1, 2, 1]), math.log(3), None),
        ("a mixed example", lambda: f(probs, labels), expected,
         "pick probs[np.arange(n), labels], then -mean(log(...))"),
        ("labels as a list", lambda: f(probs, [0, 1, 2]), expected, None),
        ("probability 0 gives a large finite loss", lambda: f(np.array([[1.0, 0.0]]), [1]), -math.log(1e-12),
         "clip with np.clip(picked, eps, 1.0) before the log"),
        ("returns a float", lambda: type(f(probs, labels)) is float, True, "wrap the result in float(...)"),
    ])


def check_bayes_posterior(f):
    _scenarios("bayes_posterior", [
        ("the lesson's example", lambda: f(0.01, 0.95, 0.90), 0.0095 / (0.0095 + 0.099), None),
        ("a common condition", lambda: f(0.5, 0.9, 0.9), 0.9, None),
        ("nobody has it", lambda: f(0.0, 0.95, 0.90), 0.0, None),
        ("no false positives means a positive is certain", lambda: f(0.01, 0.95, 1.0), 1.0, None),
        ("probability above 1 raises", lambda: f(1.2, 0.9, 0.9), Raises(ValueError), "check each input is in [0, 1]"),
        ("negative probability raises", lambda: f(0.1, -0.1, 0.9), Raises(ValueError), None),
        ("no possible positives raises", lambda: f(0.0, 0.9, 1.0), Raises(ValueError),
         "if P(positive) is 0, raise instead of dividing by zero"),
    ])


def check_bootstrap_ci(f):
    data = np.random.default_rng(7).normal(0, 1, 400)

    def width(alpha=0.05, stat=np.mean):
        low, high = f(data, stat, n_boot=3000, alpha=alpha, rng=np.random.default_rng(1))
        return high - low

    def reproducible():
        return f(data, np.median, n_boot=500, rng=np.random.default_rng(3)) == f(data, np.median, n_boot=500, rng=np.random.default_rng(3))

    def contains_mean():
        low, high = f(data, np.mean, n_boot=2000, rng=np.random.default_rng(2))
        return low < data.mean() < high

    _scenarios("bootstrap_ci", [
        ("returns two floats", lambda: [type(v) for v in f(data, np.mean, n_boot=200, rng=np.random.default_rng(0))],
         [float, float], "return float(low), float(high)"),
        ("constant data has a zero-width interval", lambda: f(np.full(30, 4.0), np.mean, n_boot=200, rng=np.random.default_rng(0)),
         (4.0, 4.0), None),
        ("resamples with replacement (the interval isn't zero width)", lambda: width() > 0.05, True,
         "without replacement every resample has the same mean; use rng.integers(0, n, size=n)"),
        ("interval for a mean is about 2 x 1.96 x sd / sqrt(n) wide", lambda: 0.17 < width() < 0.22, True,
         "check the percentiles: 100 * alpha / 2 and 100 * (1 - alpha / 2)"),
        ("contains the sample mean", contains_mean, True, None),
        ("a larger alpha gives a narrower interval", lambda: width(alpha=0.2) < width(alpha=0.05), True, None),
        ("uses the statistic it's given (a median ignores one outlier)",
         lambda: f(np.array([1.0] * 9 + [100.0]), np.median, n_boot=2000, rng=np.random.default_rng(0))[1], 1.0,
         "call statistic(resample), not np.mean"),
        ("same seed, same answer", reproducible, True, "use the rng passed in for all the randomness"),
    ])


def check_fit_linear_gd(f):
    rng = np.random.default_rng(5)
    X = rng.normal(size=(200, 3))
    y = X @ np.array([1.5, -2.0, 0.5]) + 4.0 + rng.normal(scale=0.3, size=200)
    exact = np.linalg.lstsq(np.column_stack([X, np.ones(len(X))]), y, rcond=None)[0]

    def fitted():
        w, b, _ = f(X, y, lr=0.1, epochs=500)
        return np.append(w, b)

    def losses():
        return f(X, y, lr=0.1, epochs=50)[2]

    def unchanged():
        X_copy = X.copy()
        f(X_copy, y, lr=0.1, epochs=5)
        return bool(np.array_equal(X_copy, X))

    x1 = rng.normal(size=(100, 1))
    y1 = 3 * x1[:, 0] - 1
    _scenarios("fit_linear_gd", [
        ("matches the exact least-squares answer", fitted, exact,
         "grad_w = 2 / n * X.T @ error and grad_b = 2 * mean(error), with error = X @ w + b - y"),
        ("one feature, no noise", lambda: np.append(*f(x1, y1, lr=0.1, epochs=800)[:2]), np.array([3.0, -1.0]), None),
        ("losses has one value per epoch", lambda: losses().shape, (50,), None),
        ("the first loss is mean(y squared), before any update", lambda: losses()[0], float(np.mean(y ** 2)),
         "record the loss at the start of the epoch, then update"),
        ("the loss goes down every epoch", lambda: bool(np.all(np.diff(losses()) < 0)), True, None),
        ("b is a float", lambda: type(f(X, y, epochs=3)[1]) is float, True, "return float(b)"),
        ("X is not modified", unchanged, True, None),
    ])


def check_logistic_loss_and_grad(f):
    rng = np.random.default_rng(11)
    X = rng.normal(size=(30, 3))
    y = (rng.random(30) < 0.5).astype(float)
    w = np.array([0.4, -1.2, 0.8])
    b = 0.3

    def numeric():
        h = 1e-6
        gw = np.zeros(3)
        for i in range(3):
            step = np.zeros(3)
            step[i] = h
            gw[i] = (f(w + step, b, X, y)[0] - f(w - step, b, X, y)[0]) / (2 * h)
        gb = (f(w, b + h, X, y)[0] - f(w, b - h, X, y)[0]) / (2 * h)
        return gw, gb

    def grads_match():
        _, gw, gb = f(w, b, X, y)
        nw, nb = numeric()
        return bool(np.allclose(gw, nw, atol=1e-6)) and abs(gb - nb) < 1e-6

    z = X @ w + b
    p = 1 / (1 + np.exp(-z))
    expected_loss = float(np.mean(-y * np.log(p) - (1 - y) * np.log(1 - p)))
    big = np.array([[1000.0]])

    def finite_big():
        loss0, gw0, _ = f(np.array([1.0]), 0.0, big, np.array([0.0]))
        loss1, gw1, _ = f(np.array([1.0]), 0.0, big, np.array([1.0]))
        return (round(loss0, 6), round(float(gw0[0]), 6), round(loss1, 6), round(float(gw1[0]), 6))

    _scenarios("logistic_loss_and_grad", [
        ("loss with all-zero weights is log 2", lambda: f(np.zeros(3), 0.0, X, y)[0], math.log(2), None),
        ("loss matches the formula", lambda: f(w, b, X, y)[0], expected_loss, None),
        ("gradients match a numerical check", grads_match, True,
         "dL/dz = sigmoid(z) - y; grad_w = X.T @ (p - y) / n, grad_b = mean(p - y)"),
        ("shapes and types", lambda: (type(f(w, b, X, y)[0]), f(w, b, X, y)[1].shape, type(f(w, b, X, y)[2])),
         (float, (3,), float), "return float(loss), grad_w (shape (d,)), float(grad_b)"),
        ("huge scores stay finite: (loss y=0, grad y=0, loss y=1, grad y=1)", finite_big, (1000.0, 1000.0, 0.0, 0.0),
         "use np.logaddexp(0, z) - y * z for the loss, and a stable sigmoid"),
    ])


def check_permutation_test(f):
    rng = np.random.default_rng(21)
    same_a, same_b = rng.normal(0, 1, 40), rng.normal(0, 1, 40)
    far_a, far_b = np.arange(100.0, 120.0), np.arange(0.0, 20.0)

    def unchanged():
        a, b = np.array([5.0, 1.0, 3.0]), np.array([2.0, 8.0])
        f(a, b, n_perm=50, rng=np.random.default_rng(0))
        return (a.tolist(), b.tolist())

    def reproducible():
        return f(same_a, same_b, n_perm=300, rng=np.random.default_rng(4)) == f(same_a, same_b, n_perm=300, rng=np.random.default_rng(4))

    _scenarios("permutation_test", [
        ("observed difference is mean(a) - mean(b)", lambda: f(far_a, far_b, n_perm=10, rng=np.random.default_rng(0))[0], 100.0, None),
        ("clearly different groups: p = 1 / (n_perm + 1)", lambda: f(far_a, far_b, n_perm=2000, rng=np.random.default_rng(0))[1],
         1 / 2001, "p = (count + 1) / (n_perm + 1)"),
        ("two-sided: swapping the groups gives the same tiny p", lambda: f(far_b, far_a, n_perm=2000, rng=np.random.default_rng(0))[1],
         1 / 2001, "compare abs(diff) >= abs(observed)"),
        ("small case worked out by hand: p close to 1/3",
         lambda: 0.30 < f([1.0, 2.0], [3.0, 4.0], n_perm=6000, rng=np.random.default_rng(1))[1] < 0.37, True,
         "2 of the 6 possible splits give a difference of size 2"),
        ("groups from the same distribution: p is not small",
         lambda: f(same_a, same_b, n_perm=2000, rng=np.random.default_rng(2))[1] > 0.2, True, None),
        ("inputs are not modified", unchanged, ([5.0, 1.0, 3.0], [2.0, 8.0]),
         "shuffle a pooled copy, e.g. rng.permutation(np.concatenate([a, b]))"),
        ("same seed, same answer", reproducible, True, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `stable_softmax`

In [2]:
def stable_softmax(z):
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=-1, keepdims=True)
    e = np.exp(shifted)
    return e / e.sum(axis=-1, keepdims=True)

In [3]:
check_stable_softmax(stable_softmax)

PASS  all 6 tests passed for stable_softmax


Each row needs its **own** maximum. Subtracting the overall maximum of a 2D array works for the row that contains it, but a row whose scores are all far below it turns into `exp(-999)`, which is 0, and then 0 divided by 0 is `nan`. That's what the `[[1, 2, 3], [1000, 1000, 1000]]` test catches.

`keepdims=True` keeps the reduced axis with length 1, so a `(n, k)` array minus an `(n, 1)` array of row maxima broadcasts correctly (Module 6). Without it, you'd get shape `(n,)`, which broadcasts against the columns instead, a silent bug when `n == k`.

`axis=-1` means "the last axis", so the same code handles 1D and 2D inputs. `z - ...` creates a new array, so the input isn't changed.

### Exercise 2: `log_sum_exp`

In [4]:
def log_sum_exp(z) -> float:
    z = np.asarray(z, dtype=float)
    c = z.max()
    return float(c + np.log(np.sum(np.exp(z - c))))

In [5]:
check_log_sum_exp(log_sum_exp)

PASS  all 6 tests passed for log_sum_exp


With `c = max(z)`, the largest term inside the sum is `exp(0) = 1`, so the sum is at least 1 and at most `len(z)`. It can't overflow, and its log can't be minus infinity.

The underflow case is the sneaky one. `exp(-1000)` is 0 in floating point, so the direct formula gives `log(0) = -inf`, even though the true answer is a perfectly ordinary number.

Log-sum-exp is how cross-entropy is computed from raw scores (logits): the loss for the true class `y` is `log_sum_exp(z) - z[y]`, with no softmax and no `log` of a tiny probability. SciPy has it as `scipy.special.logsumexp`, and PyTorch as `torch.logsumexp`.

### Exercise 3: `central_difference`

In [6]:
def central_difference(f, x: float, h: float = 1e-5) -> float:
    return (f(x + h) - f(x - h)) / (2 * h)

In [7]:
check_central_difference(central_difference)

PASS  all 4 tests passed for central_difference


One line, but the checks test that it really is the central difference. For `f(x) = x³` at `x = 2` with `h = 0.1`, the central difference gives `12.01` (the exact derivative is 12, and the error is exactly `h²`), while the forward difference `(f(x + h) - f(x)) / h` gives `12.61`. That difference in accuracy is the whole reason to use the central version.

A common slip is writing `/ 2 * h`, which Python reads as `(... / 2) * h`. Always put brackets around a denominator.

### Exercise 4: `cosine_similarity`

In [8]:
def cosine_similarity(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.shape != b.shape:
        raise ValueError(f"vectors have different shapes: {a.shape} and {b.shape}")
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")
    return float(a @ b / (norm_a * norm_b))

In [9]:
check_cosine_similarity(cosine_similarity)

PASS  all 8 tests passed for cosine_similarity


Without the zero check, a zero vector gives `0 / 0`, which is `nan` with a warning. A `nan` similarity then quietly breaks any sorting or ranking built on top of it, so failing loudly is better.

The length check matters too. `a @ b` would raise its own error for mismatched lengths, but a clear message saying what went wrong saves time.

In practice, embedding libraries often **normalise** vectors to length 1 when they're stored. Then cosine similarity is just a dot product, and a whole matrix of similarities is one `E @ E.T`.

## Medium

### Exercise 5: `numerical_gradient`

In [10]:
def numerical_gradient(f, x, h: float = 1e-5):
    x = np.array(x, dtype=float)          # a float copy: never touches the caller's array
    grad = np.zeros_like(x)
    for i in np.ndindex(x.shape):
        original = x[i]
        x[i] = original + h
        f_plus = f(x)
        x[i] = original - h
        f_minus = f(x)
        x[i] = original                    # restore exactly
        grad[i] = (f_plus - f_minus) / (2 * h)
    return grad

In [11]:
check_numerical_gradient(numerical_gradient)

PASS  all 5 tests passed for numerical_gradient


Three traps, each caught by a check:

- **Integers.** `np.zeros_like` of an integer array is an integer array, so a gradient of `0.5` gets truncated to `0`, and adding `h = 1e-5` to an integer element does nothing at all. `np.array(x, dtype=float)` makes a float copy.
- **Modifying the input.** `x[i] += h` then `x[i] -= h` looks like it restores the value, but it doesn't always: `0.1 + 1e-5 - 1e-5` isn't exactly `0.1` in floating point (lesson section 1). Working on a copy and saving the original value avoids both problems.
- **Any shape.** `np.ndindex(x.shape)` loops over every index tuple, `(0,)`, `(1,)` for a vector or `(0, 0)`, `(0, 1)`, ... for a matrix, so the same loop handles weight matrices.

This function does `2 × x.size` evaluations of `f`, which is why nobody trains with it. But for a gradient check on a small model it's perfect.

### Exercise 6: `gradient_descent`

In [12]:
def gradient_descent(grad, x0, lr: float, steps: int):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for _ in range(steps):
        x = x - lr * grad(x)
        path.append(x.copy())
    return np.array(path)

In [13]:
check_gradient_descent(gradient_descent)

PASS  all 6 tests passed for gradient_descent


The bug this exercise is built around: writing `x -= lr * grad(x)` and appending `x` to the path. `-=` changes the array **in place**, and the list holds references to the same array object, so every row of the path ends up equal to the final point (Module 2's aliasing, in NumPy form). Using `x = x - ...` creates a new array each step, and `.copy()` makes doubly sure.

Keeping the path, not just the final point, is what makes the charts in the lesson possible, and in real training it's the loss history you'd plot to see whether the learning rate is right.

### Exercise 7: `cross_entropy`

In [14]:
def cross_entropy(probs, labels, eps: float = 1e-12) -> float:
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels)
    picked = probs[np.arange(len(labels)), labels]     # probability of the true class, per row
    return float(-np.mean(np.log(np.clip(picked, eps, 1.0))))

In [15]:
check_cross_entropy(cross_entropy)

PASS  all 6 tests passed for cross_entropy


`probs[np.arange(n), labels]` is fancy indexing with two arrays (Module 6): it picks element `(0, labels[0])`, `(1, labels[1])` and so on, one per row. It's the standard way to select "the true class's probability" without a loop.

Two reference values worth remembering, both tested: a model that's always 100% sure and right has loss 0, and a model that predicts uniformly over `k` classes has loss `log(k)`, so 1.0986 for three classes. If your untrained model's starting loss isn't close to `log(k)`, something is wrong with the initialisation or the loss.

The clipping gives a loss of `-log(1e-12) ≈ 27.6` for a probability of 0: big enough to dominate, but finite, so training can recover.

### Exercise 8: `bayes_posterior`

In [16]:
def bayes_posterior(prior: float, sensitivity: float, specificity: float) -> float:
    for name, value in [("prior", prior), ("sensitivity", sensitivity), ("specificity", specificity)]:
        if not 0 <= value <= 1:
            raise ValueError(f"{name} must be between 0 and 1, got {value}")
    true_positives = sensitivity * prior
    false_positives = (1 - specificity) * (1 - prior)
    p_positive = true_positives + false_positives
    if p_positive == 0:
        raise ValueError("a positive result is impossible with these numbers")
    return true_positives / p_positive

In [17]:
check_bayes_posterior(bayes_posterior)

PASS  all 7 tests passed for bayes_posterior


Naming the two parts of `P(positive)` as `true_positives` and `false_positives` makes the formula easy to check against the confusion-matrix picture: the answer is the fraction of all positives that are true positives. That's exactly the **precision** of the test.

So the base-rate effect from the lesson is the same thing as "precision drops when the positive class is rare", which is why accuracy, recall and precision all need to be read together.

The edge cases are real: with `prior = 0` the answer is 0 (nobody has it), and with `specificity = 1` there are no false positives, so a positive is certain. When both terms are 0, every input is valid but the question is meaningless, and raising beats returning `nan`.

### Exercise 9: `bootstrap_ci`

In [18]:
def bootstrap_ci(data, statistic, n_boot: int = 5000, alpha: float = 0.05, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    data = np.asarray(data)
    n = len(data)
    results = np.empty(n_boot)
    for i in range(n_boot):
        results[i] = statistic(data[rng.integers(0, n, size=n)])
    low, high = np.percentile(results, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(low), float(high)

In [19]:
check_bootstrap_ci(bootstrap_ci)

PASS  all 8 tests passed for bootstrap_ci


`rng.integers(0, n, size=n)` draws `n` random row positions, with repeats allowed, which is exactly "resample with replacement". Indexing with positions instead of calling `rng.choice` on the data also works for 2D data, where each row (one example) must stay together.

The tests check behaviour rather than exact numbers:

- **Without replacement**, every resample is just the data reordered, so the mean never changes and the interval has zero width. That's the most common bootstrap bug.
- **For a mean**, the interval should be about `2 × 1.96 × σ / sqrt(n)` wide, the textbook answer, and the bootstrap gets there without any formula.
- **A larger `alpha`** (less confidence) gives a narrower interval.

Passing `rng` in, instead of using a global seed, keeps functions reproducible and independent: two calls with generators seeded the same way give the same interval.

## Hard

### Exercise 10: `fit_linear_gd`

In [20]:
def fit_linear_gd(X, y, lr: float = 0.1, epochs: int = 500):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = np.empty(epochs)
    for epoch in range(epochs):
        error = X @ w + b - y               # shape (n,)
        losses[epoch] = np.mean(error ** 2)
        grad_w = 2 / n * (X.T @ error)      # shape (d,)
        grad_b = 2 * np.mean(error)
        w = w - lr * grad_w
        b = b - lr * grad_b
    return w, float(b), losses

In [21]:
check_fit_linear_gd(fit_linear_gd)

PASS  all 7 tests passed for fit_linear_gd


The gradients, by the chain rule. With `e = X @ w + b - y` and `L = mean(e²)`:

- `dL/de = 2e / n` (one term per example)
- `de/dw = X` and `de/db = 1`
- so `dL/dw = 2/n · Xᵀe` and `dL/db = 2 · mean(e)`

`X.T @ error` is the shape-checking trick from the lesson: `(d, n) @ (n,)` gives `(d,)`, one number per weight, which is what a gradient for `w` has to be. If the shapes don't work out, the formula is wrong.

The first recorded loss is `mean(y²)`, because the model starts by predicting 0 for everything. The checks use that as an independent test that the loss is recorded before the update.

The test data is standardised, so `lr = 0.1` is safe. On raw features with very different scales, the same learning rate would either crawl or blow up, exactly as in the lesson's stretched bowl.

### Exercise 11: `logistic_loss_and_grad`

In [22]:
def sigmoid(z):
    # 1 / (1 + exp(-z)) never overflows when z >= 0; exp(z) / (1 + exp(z)) never overflows when z < 0
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1 + ez)
    return out


def logistic_loss_and_grad(w, b: float, X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    z = X @ np.asarray(w, dtype=float) + b
    loss = np.mean(np.logaddexp(0, z) - y * z)
    error = sigmoid(z) - y                    # dL/dz for each example, before averaging
    grad_w = X.T @ error / len(y)
    grad_b = np.mean(error)
    return float(loss), grad_w, float(grad_b)

In [23]:
check_logistic_loss_and_grad(logistic_loss_and_grad)

PASS  all 5 tests passed for logistic_loss_and_grad


**Where `log(1 + exp(z)) - y·z` comes from.** Since `p = 1 / (1 + e^-z)`, we have `log p = -log(1 + e^-z)` and `log(1 - p) = -z - log(1 + e^-z)`. Substituting and simplifying gives `log(1 + e^z) - y·z`. This form never takes the log of a number that rounded to 0.

**The gradient.** The derivative of `log(1 + e^z) - y·z` with respect to `z` is `sigmoid(z) - y`. Everything else in the sigmoid and the log cancels. Then `dz/dw = x` gives `grad_w = Xᵀ(p - y) / n`, the same shape as linear regression's gradient with `p - y` in place of the error. This tidy result is why sigmoid and log loss are always used together.

**The stable sigmoid** picks one of two equivalent formulas depending on the sign of `z`, so `exp` only ever sees a negative number and can't overflow. `scipy.special.expit` does the same.

This function is the core of the mini project, where you'll train it, check it and analyse it.

### Exercise 12: `permutation_test`

In [24]:
def permutation_test(a, b, n_perm: int = 10_000, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])           # a new array, so shuffling it can't touch a or b
    count = 0
    for _ in range(n_perm):
        shuffled = rng.permutation(pooled)
        diff = shuffled[:len(a)].mean() - shuffled[len(a):].mean()
        if abs(diff) >= abs(observed):
            count += 1
    return float(observed), (count + 1) / (n_perm + 1)

In [25]:
check_permutation_test(permutation_test)

PASS  all 7 tests passed for permutation_test


**Two-sided** means "a difference this big in either direction". Without `abs`, a test of `a` against `b` and a test of `b` against `a` would give completely different p-values, and the check that swaps them catches it.

**Why `+ 1`?** The real labelling is one of the possible arrangements, so it counts as a shuffle that's at least as extreme as itself. It also means the p-value is never 0. With 10,000 shuffles, the smallest possible value is about 0.0001, which honestly reports "rarer than 1 in 10,000" instead of "impossible".

The small test case can be worked out by hand. For `a = [1, 2]` and `b = [3, 4]`, there are 6 ways to pick which two values form group `a`, and 2 of them, `{1, 2}` and `{3, 4}`, give a difference of size 2. So the p-value should be close to 1/3.

`rng.permutation(pooled)` returns a shuffled **copy**. `rng.shuffle` shuffles in place, which is fine on `pooled` but would scramble the caller's data if used on `a` directly.

A vectorised version (shuffle a 2D array of indices at once) is much faster, but the loop is easier to read and fast enough for a few thousand shuffles.

## What's next

The mini project in `04_mini_project.ipynb` builds logistic regression from scratch, checks its gradient, trains it two different ways, and measures how much its test accuracy can be trusted.